In [10]:
import os
from langchain_groq import ChatGroq
from userSecrets import groq_secret
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
os.environ['GROQ_API_KEY']=groq_secret

In [11]:
llm = ChatGroq(model='meta-llama/llama-4-scout-17b-16e-instruct',max_tokens=2000,temperature=0.4)

In [12]:
loader=PyPDFLoader('Kunal_s_resume (13).pdf')
doc=loader.load()

In [13]:
template=PromptTemplate(
    input_variables=['resume'],
    template='''Analyse and return Strengths and Weaknesses of the resume:
    {resume} 
    in strict json format like this
    {format}
    ''',
    partial_variables={'format':JsonOutputParser().get_format_instructions()}
)



In [14]:
chain=template|llm|JsonOutputParser()

In [15]:
res=chain.invoke({"resume":doc[0].page_content})

In [16]:
res["Strengths"]

['Strong problem-solving background with experience in competitive programming (Codeforces max at 1457, CodeChef max at 1704)',
 'Hands-on experience in Full Stack Development and Machine Learning',
 'Proficiency in a range of technical skills, including languages (C++, Python, Javascript), frontend (React, Next.js), backend (Node.js, Express.js), and ML/AI (scikit-learn, Pandas, NumPy)',
 'Experience with modular ML pipelines and GenAI applications using LangChain and LLM APIs',
 'Portfolio of projects showcasing practical applications of skills, including ML/GenAI and web development projects',
 'Strong educational background with a high aggregate CGPA (8.77) at DTU']

In [17]:
res["Weaknesses"]

['Lack of clear career goals or objective statement',
 'Limited work experience (only a 3rd-year student)',
 'No explicit mention of soft skills, such as communication, teamwork, or leadership',
 'Interests section is brief and limited to only two items (exercise and poetry)',
 'No mention of any relevant certifications or achievements outside of academic projects',
 'Some project descriptions could be more detailed, and impact/achievements could be quantified (e.g., number of users, metrics, etc.)']

In [ ]:
from pydantic import BaseModel
from typing import List

class ResponseSchema(BaseModel):
    preamble:str
    size:int
    res:List[str]

from langchain_core.output_parsers import PydanticOutputParser
parser=PydanticOutputParser(pydantic_object=ResponseSchema)
userPromptTemplate=PromptTemplate(
    input_variables=['prompt','context'],
    template='''{prompt}. Respond in crisp and concise points int 200-300 tokens only
    context:{context}
        formt={format}
    '''  ,
    partial_variables={
        "format":parser.get_format_instructions()
    }
)


In [53]:
chain2=userPromptTemplate|llm|parser

In [62]:
resume=doc[0].page_content
def sectionTagger(text:str):
    sections={
        "education":"",
        "projects":"",
        "skills":"",
        "experience":""
    }

    currentSec=None
    for line in text.split('\n'):
        lowerline=line.lower().strip()
        if "education" in lowerline.split(" ")[0].replace(":",""):
            currentSec="education"
            
        elif "projects" in lowerline.split(" ")[0].replace(":",""):
            currentSec="projects"
            
        elif "skills" in lowerline.split(" ")[0].replace(":","") or "technical" in lowerline.split(" ")[0].replace(":",""):
            currentSec="skills"
            
        elif "experience" in lowerline.split(" ")[0].replace(":",""):
            currentSec="experience"
            

        if currentSec:
            sections[currentSec]+=line+"\n"
    return sections

sections=sectionTagger(resume)

def getSections(text:str):
        text=text.lower()
        sectionsInPrompt=[]
        if any(word in text for word in ["education", "college", "degree", "study"]):
            sectionsInPrompt.append("education")

        if any(word in text for word in ["project", "projects"]):
            sectionsInPrompt.append("projects")

        if any(word in text for word in ["skill", "skills", "tech", "stack"]):
            sectionsInPrompt.append("skills")

        if any(word in text for word in ["experience", "work", "intern"]):
            sectionsInPrompt.append("experience")

        return sectionsInPrompt

prompt="Tell me about my education, projects and skills"

contextSec=getSections(prompt)

context=""
for sec in contextSec:
    context += f"{sec.upper()}:\n{sections[sec]}\n\n"
if context=="":
    context=resume


In [64]:
res=chain2.invoke({"prompt":prompt,"context":context})

In [66]:
res.preamble

'Your Education, Projects, and Skills Summary'

In [67]:
res.size

3

In [69]:
res.res

['Completed a degree in Computer Science with a strong foundation in programming languages, data structures, and software engineering',
 'Worked on various projects, including a web development project using React, Node.js, and MongoDB, and a machine learning project using Python and TensorFlow',
 'Proficient in programming languages such as Python, Java, JavaScript, and C++, with experience in data analysis, problem-solving, and teamwork']